현재 구성에서 벡터 DB의 필요성은 높습니다. 다만 Neo4j를 대체하는 저장소가 아니라, 자연어 질문과 그래프 노드 사이의 의미적 연결점을 찾는 1차 검색 계층으로 도입해야 합니다.

## 1. 현재 방식의 한계

현재 엔티티 확인은 [text2cypher.py](<C:/Users/Playdata/Desktop/김동석/교과목-2/mle-01-p2-team3/src/agent/tools/text2cypher.py>)에서 다음과 같은 정확 일치 방식으로 동작합니다.

```cypher
MATCH (n)
WHERE n.name = $name
RETURN n
```

따라서 아래 질문은 잘 처리할 수 있습니다.

- “롯데쇼핑의 계열회사는?”
- “한솔홀딩스가 가진 종속기업은?”
- “서울에 위치한 기업은?”

반면 다음과 같은 자연어 질문에서는 검색 기준점을 찾기 어렵습니다.

- “태양광 관련 사업을 하는 전남 지역 기업”
- “자동차 부품을 만드는 회사와 연결된 그룹사”
- “롯데건설과 이름이나 사업이 비슷한 기업”
- “배터리 소재 공급 후보로 볼 만한 회사”
- “한솔제지와 연관 있는 회사”처럼 표기나 문맥이 불명확한 질문

현재 그래프 조회는 사용자가 말한 표현이 Neo4j의 `name`과 정확히 일치해야 안정적으로 시작할 수 있습니다. 벡터 검색이 없으면 `business_content`, 업종명, 정규화 이름, 주소 등의 의미를 이용해 시작 노드를 찾기 어렵습니다.

## 2. 벡터 DB가 필요한 이유

| 필요성 | 현재 문제 | 벡터 검색의 역할 |
| --- | --- | --- |
| 의미 기반 검색 | “태양광 발전”, “신재생에너지 사업”처럼 표현이 다름 | 의미가 가까운 기업과 업종을 검색 |
| 기업명 표기 차이 | `(주)`, `주식회사`, 영문명, 공백 차이 존재 | 질문과 유사한 이름 후보를 검색 |
| 모호한 질문 처리 | 질문에 정확한 기업명이 없을 수 있음 | 관련 기업·업종을 시작 노드로 제시 |
| 종속기업 식별 | 종속기업은 법인등록번호가 없음 | 이름·주소·사업 내용으로 후보 검색 |
| 유사 기업 추천 | 그래프 관계만으로 사업 유사성을 판단하기 어려움 | 사업 내용과 업종 임베딩을 비교 |
| 긴 근거 검색 | 향후 공시·뉴스·기업 설명이 추가될 수 있음 | 관련 문서 조각을 검색해 근거 제공 |

특히 현재 데이터에는 7,144개의 `SubsidiaryCompany` 노드가 있고, `business_content`와 `name_norm` 같은 의미 검색에 적합한 속성이 있습니다. 이 정보를 단순 정확 일치로만 사용하는 것은 활용도가 낮습니다.

## 3. 벡터 DB가 불필요한 경우

모든 질의에 벡터 검색이 필요한 것은 아닙니다.

| 질문 | 적합한 검색 |
| --- | --- |
| 법인등록번호가 `1101110000086`인 기업 | 정확 조회 |
| 롯데쇼핑의 직접 계열회사 | Neo4j 관계 조회 |
| 특정 기업의 종속기업 수 | Cypher 집계 |
| 서울 지역 기업 수 | Cypher 필터·집계 |
| 두 기업 사이의 최단 관계 경로 | Neo4j 경로 탐색 |

관계, 경로, 개수, 그룹 구조처럼 정형화된 질문은 Neo4j가 더 정확합니다. 벡터 DB는 질문을 이해하고 적절한 시작 노드를 찾는 데 사용하고, 최종 관계 계산은 Neo4j가 담당해야 합니다.

## 4. 권장 아키텍처

현재 프로젝트에는 다음과 같은 2단계 하이브리드 검색이 적합합니다.

```text
사용자 질문
    ↓
질문 정규화 및 임베딩
    ↓
1차: Vector Anchor Retrieval
기업·종속기업·업종 후보 Top-K 검색
    ↓
후보 ID 확정
    ↓
2차: Neo4j Cypher Graph Expansion
계열·종속·지역·업종 관계 확장
    ↓
관계 경로와 evidence 수집
    ↓
LLM 답변 생성
```

핵심은 벡터 검색 결과에서 기업 이름만 받는 것이 아니라 Neo4j 노드의 고유 ID를 받는 것입니다. 그 ID를 파라미터화된 Cypher에 전달해야 안정적으로 그래프를 확장할 수 있습니다.

## 5. 1차 Vector Anchor Retrieval

### 인덱싱 대상

우선 다음 두 노드 유형만 벡터화하는 것이 적절합니다.

| 노드 | 임베딩할 정보 | 메타데이터로 보관할 정보 |
| --- | --- | --- |
| `ParentCompany` | 기업명, 주소, 업종, 지역 | `id`, `crno`, 노드 유형 |
| `SubsidiaryCompany` | 기업명, 정규화 이름, 주소, 주요 사업 내용, 국내 여부 | `id`, 노드 유형, `domestic` |

`Region`과 `Industry`는 각각 17개, 301개로 규모가 작고 명칭이 비교적 정형화되어 있습니다. 초기에는 정확 일치·동의어 사전·필터로 처리하고, 필요할 때 벡터 인덱스에 추가하는 편이 효율적입니다.

### 검색 문서 구성 예

```text
기업명: 한솔페이퍼텍
정규화 이름: 한솔페이퍼텍
주소: 전라남도 담양군 ...
지역: 전남
주요 사업: 제지 제조 및 판매
국내 여부: 국내
```

법인등록번호는 의미 임베딩의 대상이 아니라 검색 결과를 그래프 노드에 연결하는 메타데이터로 보관합니다.

### 청킹 기준

현재 기업 데이터는 한 노드당 텍스트가 짧으므로 문서 청킹이 필요하지 않습니다.

- 기업 데이터: 한 기업당 한 임베딩
- 업종 데이터: 한 업종당 한 임베딩
- 향후 뉴스·공시: 문단이나 의미 단위로 청킹
- 관계 근거: 원천 행 또는 근거 문장 단위로 청킹

## 6. 2차 Neo4j Graph Expansion

벡터 검색이 반환한 노드 ID를 사용해 관계를 확장합니다.

예를 들어 “태양광 관련 국내 기업과 그 모기업”이라는 질문은 다음처럼 처리합니다.

1. `business_content`가 태양광과 의미적으로 가까운 종속기업을 검색합니다.
2. 검색된 종속기업 ID를 확보합니다.
3. Neo4j에서 `HAS_SUBSIDIARY` 관계를 역방향으로 조회합니다.
4. 모기업의 지역·업종·계열회사까지 필요한 깊이만 확장합니다.
5. 관계의 `evidence`, `source_case`, `source_row`를 답변 근거로 사용합니다.

개념적인 파라미터화 쿼리는 다음과 같습니다.

```cypher
MATCH (parent:ParentCompany)-[r:HAS_SUBSIDIARY]->(sub:SubsidiaryCompany)
WHERE sub.id IN $anchor_ids
RETURN
    parent.id AS parent_id,
    parent.name AS parent_name,
    sub.id AS subsidiary_id,
    sub.name AS subsidiary_name,
    r.evidence AS evidence,
    r.source_case AS source_case,
    r.source_row AS source_row
LIMIT $limit
```

LLM이 생성한 사용자 문자열을 Cypher에 직접 삽입하지 않고, `$anchor_ids`와 `$limit` 같은 파라미터로 전달해야 합니다.

## 7. 저장 방식 선택

현재 규모에서는 별도의 벡터 DB 서버를 추가하기보다 Neo4j Vector Index를 우선 검토하는 것이 적합합니다.

| 방안 | 장점 | 단점 | 적합성 |
| --- | --- | --- | --- |
| Neo4j Vector Index | 그래프 ID와 벡터가 같은 DB에 존재, 운영 구조가 단순함 | 대규모 벡터 전용 기능은 제한될 수 있음 | 현재 프로젝트에 가장 적합 |
| 외부 벡터 DB | 검색 기능과 확장성이 좋음 | Neo4j ID 동기화와 별도 운영 필요 | 데이터·트래픽 증가 후 검토 |
| 로컬 FAISS | 실험이 간단함 | 영속성, 동시 접근, 운영 관리가 어려움 | 초기 성능 실험용 |

현재 그래프는 노드 8,324개 규모이므로 별도 벡터 인프라를 먼저 도입할 실익이 크지 않습니다. Neo4j 안에서 벡터 앵커와 그래프 관계를 함께 관리하는 구성이 단순합니다.

## 8. 권장 모듈 구조

```text
src/
├── retrieval/
│   ├── embedding.py          # 검색 텍스트와 임베딩 생성
│   ├── vector_store.py       # 벡터 인덱스 생성·갱신·검색
│   └── anchor_retriever.py   # Top-K 앵커와 점수 반환
├── graph/
│   ├── graph_db.py           # Neo4j 연결과 파라미터화 쿼리
│   └── graph_expander.py     # 앵커 기준 관계 확장
├── rag/
│   ├── context_builder.py    # 벡터·그래프 결과 통합
│   └── chain.py              # 근거 기반 답변 생성
└── streamlit/
    └── chatbot.py
```

현재 [text2cypher.py](<C:/Users/Playdata/Desktop/김동석/교과목-2/mle-01-p2-team3/src/agent/tools/text2cypher.py>)가 Neo4j 연결, 온톨로지 로딩, 엔티티 조회, 임의 Cypher 실행을 함께 담당합니다. 벡터 검색을 추가할 때는 Neo4j 연결과 쿼리를 `graph_db.py`로 분리하는 것이 안전합니다.

## 9. 구축 단계

### 1단계: 검색 데이터 정제

- `case=3`의 `top_*` 복수값을 같은 위치 기준으로 분리
- 종속기업 중 `name_norm + address` 중복 검사
- 빈 주소와 사업 내용 처리
- 노드별 검색용 문자열 생성
- 원본 노드 ID와 검색 문서의 1:1 연결 보장

### 2단계: 임베딩 및 인덱스 구축

- 임베딩 모델과 차원 고정
- `ParentCompany`, `SubsidiaryCompany` 임베딩 생성
- 노드 ID, 라벨, 모델명, 임베딩 버전 저장
- 동일 데이터 재실행 시 중복 생성되지 않도록 upsert 적용

### 3단계: 앵커 검색 구현

검색 결과는 최소한 다음 구조를 가져야 합니다.

```json
{
  "anchor_id": "subsidiary:...",
  "node_type": "SubsidiaryCompany",
  "name": "기업명",
  "score": 0.87
}
```

- Top-K 결과 반환
- 최소 유사도 기준 적용
- 국내 여부·지역·노드 유형 필터 지원
- exact match와 `name_norm` 일치는 점수를 높이는 방식으로 결합

### 4단계: 그래프 확장 구현

- 허용된 질문 유형별 Cypher 템플릿 정의
- 모든 값 파라미터화
- 관계 깊이와 결과 개수 제한
- `evidence`, `source_case`, `source_row` 포함
- 읽기 전용 Neo4j 계정 사용

### 5단계: 답변 생성

LLM에는 원본 그래프 전체가 아니라 다음만 전달합니다.

- 사용자 질문
- 선택된 앵커와 유사도
- 조회된 관계 경로
- 관련 기업 속성
- 관계 근거
- 데이터가 없는 항목과 조회 제한

## 10. 평가 방법

벡터 DB를 구축했다고 해서 검색 품질이 자동으로 좋아지는 것은 아닙니다. 최소한 다음 지표가 필요합니다.

| 단계 | 평가 지표 |
| --- | --- |
| 앵커 검색 | Recall@K, MRR, 정확한 기업 포함 여부 |
| 엔티티 식별 | 동명 기업 구분 정확도, 표기 변형 매칭률 |
| 그래프 확장 | 올바른 관계·방향·깊이 조회 비율 |
| 답변 | 근거 일치율, 관계 정확도, 미지원 질문 거절률 |
| 운영 | 응답 시간, 토큰 비용, 임베딩 갱신 시간 |

평가 질문은 유형별로 나누는 것이 좋습니다.

- 정확한 기업명 질문
- 약칭·오타·법인 표기 차이가 있는 질문
- 사업 내용 기반 질문
- 지역과 업종이 결합된 질문
- 다단계 관계 질문
- 그래프에 답이 없는 질문

## 11. 주요 위험과 대응

| 위험 | 대응 |
| --- | --- |
| 유사하지만 다른 기업을 앵커로 선택 | Top-K 후보와 점수를 표시하고 임계값 미달 시 재질문 |
| 같은 `name_norm`을 가진 기업 | 주소, 모기업 관계, 법인등록번호를 추가 확인 |
| 오래된 관계를 현재 관계처럼 답변 | 기준일 필드를 추가하기 전에는 시점 한계를 명시 |
| 벡터 검색 결과만으로 관계를 추정 | 실제 관계는 반드시 Neo4j Cypher 결과로 확인 |
| LLM 생성 Cypher의 보안 문제 | `graph_db.py`의 파라미터화된 조회 템플릿과 읽기 전용 계정 사용 |
| 임베딩 모델 변경으로 점수 불일치 | 모델명·차원·버전을 저장하고 전체 재색인 |
| 데이터와 인덱스 불일치 | 노드 데이터 해시 또는 갱신 시각을 저장하고 증분 동기화 |

## 결론

현재 구조에서도 정확한 기업명과 관계가 주어지면 Neo4j만으로 질의가 가능합니다. 그러나 자연어 기반 기업 탐색, 사업 내용 검색, 이름 변형 처리, 유사 기업 추천까지 제공하려면 벡터 DB가 필요합니다.

권장 방향은 다음과 같습니다.

1. Neo4j Vector Index로 기업·종속기업의 검색 앵커를 구축합니다.
2. 질문에서 Top-K 기업 ID를 찾습니다.
3. 해당 ID를 파라미터화된 Cypher에 전달합니다.
4. Neo4j에서 실제 관계만 확장합니다.
5. 관계 근거와 함께 답변합니다.

즉, 벡터 DB는 “답을 만드는 DB”가 아니라 “그래프에서 어디부터 찾아야 하는지 결정하는 DB”로 사용하는 것이 적합합니다.